## 1. Read data

### 1.1. The dataset of sentences

In [1]:
import pandas as pd
sentence_df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_datasets/translated_sentence_data.xlsx")

In [2]:
sentences = sentence_df["Clean_Sentence"].to_list()

### 1.2. Import list of stopwords

In [197]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/mijnidbcoachnlp/data/analysis_datasets/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec', "jl"
]

dutch_stopwords.extend(extra_list)

## 2. Download the embedding model BERTje

### 2.1 Download BERTje

In [4]:
from transformers.pipelines import pipeline
from transformers import AutoTokenizer, AutoModel, TFAutoModel
embedding_model = pipeline("feature-extraction", model="GroNLP/bert-base-dutch-cased")

Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 2.2 Pre-calculate and save sentence embeddings (skip if there are saved embeddings and go to 2.3)

In [5]:
# pre-calculate the embeddings of the dutch sentences
import numpy as np

embeddings_nl = embedding_model(sentences, truncation=True, padding=True)
sentence_embeddings_nl = np.array([np.mean(embedding, axis=1).flatten() for embedding in embeddings_nl])

KeyboardInterrupt: 

In [8]:
from tqdm import tqdm 
# Initialize an empty list to store embeddings
sentence_embeddings_nl = []

# Use tqdm to track progress
for sentence in tqdm(sentences, desc="Processing Embeddings", unit="sentence"):
    # Compute embedding for the sentence
    embedding = embedding_model(sentence)
    # Average token embeddings to create a sentence-level embedding
    sentence_embedding = np.mean(embedding[0], axis=0)
    # Append to the list
    sentence_embeddings_nl.append(sentence_embedding)

# Convert the list to a NumPy array
sentence_embeddings_nl = np.array(sentence_embeddings_nl)

# Check the shape of the resulting embeddings
print(f"Shape of sentence embeddings: {sentence_embeddings_nl.shape}")

Processing Embeddings: 100%|██████████| 42537/42537 [1:35:47<00:00,  7.40sentence/s]


Shape of sentence embeddings: (42537, 768)


In [10]:
# save the sentence embeddings
import numpy as np
np.save('/workspace/mijnidbcoachnlp/data/embeddings/sentence_embeddings_bertje_nl_new.npy', sentence_embeddings_nl)

### 2.3 Load saved embeddings

In [228]:
# load the save model
import numpy as np

loaded_embeddings_nl = np.load('/workspace/mijnidbcoachnlp/data/embeddings/sentence_embeddings_bertje_nl.npy')

In [229]:
embeddings=loaded_embeddings_nl

## 3. Preparation for Fine-Tuning Hyperparameters

### 3.1. Generate a list of hyperparameter combinations

In [16]:
import itertools

# Generating a list of hyperparameter combinations
# UMAP hyperparameters
n_neighbors_values = [5, 10, 15, 20, 25, 30] # we are aiming for strict clusters so the n_neighbors range is relatively small
n_components_values = [2, 3, 4, 5]

# HDBSCAN hyperparameters
min_cluster_size_values = [10, 15, 20, 30]
min_samples_values = [5, 10, 15, 20, 30]

combinations = list(itertools.product(n_neighbors_values, n_components_values, min_cluster_size_values, min_samples_values))

# set min_samples < min_cluster_size
filtered_combinations = [
    combination for combination in combinations if combination[3] <= combination[2] 
]

# print the total number of combinations
len(filtered_combinations)

336

### 3.2 Function to calculate intra-topic similarity

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# function to calculate intra-topic cosine similarity
def calculate_intra_topic_cosine_similarity(embeddings, doc_topics):
    topic_similarities = []
    
    for topic in set(doc_topics):
        if topic == -1:  # Skip the "outlier" topic
            continue
        
        # Get embeddings of documents in the current topic
        topic_embeddings = embeddings[np.array(doc_topics) == topic]
        
        if len(topic_embeddings) < 2:
            continue  # Skip topics with a single document
        
        # Calculate cosine similarities and average them
        cosine_sim = cosine_similarity(topic_embeddings)
        avg_cosine_sim = cosine_sim[np.triu_indices_from(cosine_sim, k=1)].mean()
        
        topic_similarities.append(avg_cosine_sim)

    return topic_similarities

### 3.3. Function to calculate silhouette scores

In [9]:
# function to calculate silhouette scores if needed (we don't use the silhouette scores for now because the clusters are probably overlapping a lot so inter-topic distance is not so informative)
from sklearn.metrics import silhouette_samples
import numpy as np

def calculate_intra_topic_silhouette_score(embeddings, topics):

    # Filter out documents assigned to the outlier topic (-1)
    mask = topics != -1
    filtered_embeddings = embeddings[mask]
    filtered_topics = topics[mask]

    # Compute silhouette scores
    silhouette_scores = silhouette_samples(filtered_embeddings, filtered_topics, metric="cosine")
    
    # Calculate average silhouette score for each topic
    unique_topics = np.unique(filtered_topics)
    avg_silhouette_per_topic = {
        topic: np.mean(silhouette_scores[filtered_topics == topic]) for topic in unique_topics
    }

    # Compute the overall average silhouette score across all topics
    avg_silhouette_score = np.mean(list(avg_silhouette_per_topic.values()))

    return avg_silhouette_score


### 3.4 Function to calculate coherence scores

In [10]:
# create the dictionary of documents
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

def generate_dictionary(documents):

    processed_docs = [doc.split() for doc in documents]
    
    # Create a Gensim dictionary
    dictionary = Dictionary(processed_docs)
    
    return dictionary, processed_docs

dictionary, processed_docs = generate_dictionary(sentences)

In [11]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

def generate_topic_words(topics, top_n_words):
    
    topic_words = []
    
    for topic_num, words in topics.items():
        if topic_num == -1:  # Skip the outlier topic
            continue
        # Collect only the top N words for each topic
        top_words = [word for word, _ in words[:top_n_words]]
        topic_words.append(top_words)
    
    return topic_words

def calculate_coherence_score(topics, dictionary, processed_docs, coherence='c_v', top_n_words=10):

    topic_words = generate_topic_words(topics, top_n_words)

    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=processed_docs,
        dictionary=dictionary,
        coherence=coherence
    )
    
    # Step 4: Get the coherence score
    coherence_score = coherence_model.get_coherence()
    
    return coherence_score


## 3. Preparation for Model Evaluation

### 3.1 Import the annotated sample set

In [46]:
annotated_df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_datasets/sample_sentences_for_labeling_100.xlsx", index_col=0)

In [47]:
annotated_df #N = 328 sentences

,Message_ID,Sentence,Clean_Sentence,Sentence_ID,Translated_Sentence,Label_Specific_Jiaxu
269,173,"Beste [PERSOON-1], het gaat weer iets beter ...",het gaat weer iets beter en ik heb gisteren de...,269,"Dear [PERSOON-1], things are getting better an...",Tests
270,173,Het liep goed totdat ik probeerde de onderkant...,Het liep goed totdat ik probeerde de onderkant...,270,It went well until I tried to break the bottom...,Tests
271,173,Ik kreeg het niet voor elkaar en heb toen meer...,Ik kreeg het niet voor elkaar en heb toen meer...,271,I couldn't do it and then I used more power.,Other
272,173,Hij lijkt me nu op de verkeerde plek gebroken ...,j lijkt me nu op de verkeerde plek gebroken (z...,272,It looks like it's broken in the wrong place (...,Other
273,173,De test is uiteindelijk mislukt volgens de app.,De test is uiteindelijk mislukt volgens de app.,273,The test ultimately failed according to the app.,Tests
...,...,...,...,...,...,...
42370,33048,"Omdat ik aangaf dat het echt niet goed ging, h...","Omdat ik aangaf dat het echt niet goed ging, h...",42370,"Because I indicated that it was really bad, sh...",Complaint_Other_Symptom
42371,33048,De dienstdoendearts gaf aan dat de echo goed w...,De dienstdoendearts gaf aan dat de echo goed w...,42371,The medical examiner indicated that the ultras...,Examination
42519,33204,"Hoi [PERSOON-1], Dank voor je snelle reactie:...",Dank voor je snelle reactie:) Ik zal een zelf...,42519,"Hi [PRESSON-1], Thank you for your quick react...",Tests
42520,33204,"Dank voor de afspraak in september, staat geno...","Dank voor de afspraak in september, staat geno...",42520,"Thanks for the September appointment, it's noted.",Appointment_Info


In [50]:
# print the unique labels
unique_labels = set(annotated_df["Label_Specific_Jiaxu"].to_list())
unique_labels

{'Advice_Question',
 'App',
 'Appointment_Info',
 'Appointment_Reference',
 'Appointment_Scheduling',
 'Complaint_Gastrointestinal_Symptom',
 'Complaint_Other_Symptom',
 'Contact',
 'Covid',
 'Diet',
 'Discussion',
 'Distress',
 'Document_Request',
 'Examination',
 'External_Care',
 'Hospital_Visit',
 'Improvement',
 'Medication_Biologics_Vaccination',
 'Medication_Effect',
 'Medication_Use',
 'Other',
 'Patient_Availability',
 'Patient_Health_Update',
 'Patient_Info',
 'Pharmacy_Info',
 'Pharmacy_Prescription',
 'Pregnancy',
 'Survey',
 'Test_Result_Inquiry',
 'Test_Result_Report',
 'Tests',
 'Weight',
 'Work'}

In [51]:
len(unique_labels)
# there are 34 unique labels of this annotated sample set

33

### 3.2 Function to build model with different clustering methods

In [221]:
# set the vectorizer model
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from hdbscan import HDBSCAN
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired



# configure the sub-models
#vectorizer
vectorizer_model=CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 2), token_pattern=r'\b[a-zA-Z]{2,}\b')
# clustering (3 options)
hdb_model = HDBSCAN(min_cluster_size=20, min_samples=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
kmeans_model = KMeans(n_clusters=100)
agglo_model = AgglomerativeClustering(n_clusters=100)
# dimension reduction
umap_model=UMAP(n_neighbors=10, n_components=2, metric='cosine', random_state=42)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
representation_model = KeyBERTInspired()


In [230]:
# function to build the model with different selection of clustering method
# build the BERTopic model
from bertopic import BERTopic
from umap import UMAP
from bertopic import BERTopic


def build_bertopic(clustering_method):
    cluster_model=hdb_model
    if clustering_method == "KMeans":
        cluster_model = KMeans(n_clusters=50)
    
    if clustering_method == "Agglomerative":
        cluster_model = AgglomerativeClustering(n_clusters=50)
    

    # Initialize BERTopic with UMAP and HDBSCAN models
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        top_n_words=10,
        verbose=True
    )

    # Fit-transform to get document topics and probabilities
    topics, probs = topic_model.fit_transform(sentences, embeddings)
    

    # Return only essential results
    return topic_model

## 4. BERTje-HDBSCAN model

In [231]:
topic_model = build_bertopic(clustering_method = "HDBSCAN")

2024-11-17 03:35:51,816 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-11-17 03:36:40,496 - BERTopic - Dimensionality - Completed ✓
2024-11-17 03:36:40,500 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-11-17 03:36:42,439 - BERTopic - Cluster - Completed ✓
2024-11-17 03:36:42,454 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-11-17 03:36:43,625 - BERTopic - Representation - Completed ✓


In [232]:
topic_model.get_topic_info() #this is 20, 10 of HDBSCAN

,Topic,Count,Name,Representation,Representative_Docs
0,-1,8726,-1_recept_graag_goed_nieuw,"[recept, graag, goed, nieuw, apotheek, uitslag, weten, gaat, sturen, nieuw recept]","[Alvast bedankt voor de moeite., Kunt u een nieuw recept naar apotheek voor mij sturen?, Kunt u mij zodra de uitslagen binnen zijn overigens een nieuw recept sturen naar mijn apotheek: Apotheek Alvast bedankt en]"
1,0,22984,0_ontlasting_afspraak_weken_week,"[ontlasting, afspraak, weken, week, last, bloed, pijn, keer, gaan, dagen]","[Ik heb deze ochtend bloed en ontlasting ingeleverd in ., Ik heb vorige week 1 malig bloed gehad bij de ontlasting erna noit meer gezien., Ik heb maandag 19-09 bloed laten prikken en ontlasting ingeleverd.]"
2,1,481,1_ziekenhuis_apotheek ziekenhuis_ziekenhuis ziekenhuis_apotheek,"[ziekenhuis, apotheek ziekenhuis, ziekenhuis ziekenhuis, apotheek, zorginstelling, ziekenhuis apotheek, ziekenhuis recept, prikken ziekenhuis, ziekenhuis sturen, afspraak ziekenhuis]","[Vanuit ziekenhuis, Gewoon in het [ZIEKENHUIS]., Beide bij het [ZIEKENHUIS] in , .]"
3,2,367,2_snelle_bedankt_reactie_snelle reactie,"[snelle, bedankt, reactie, snelle reactie, bedankt snelle, dank, bedankt reactie, dankjewel, reactie dank, reactie bedankt]","[Oke, bedankt voor de snelle reactie., Bedankt voor je snelle reactie!, Bedankt voor de snelle reactie!]"
4,3,330,3_vraag_overleg_infusie_ter,"[vraag, overleg, infusie, ter, huisarts, schrijven, zaken, nadelen, omgaand reactie, ontvangen bevolkingsonderzoek]","[Daarom overleg ik graag vandaag nog (d.w.z., Ter info voor het overleg v.w.b., Ik heb vandaag terugkoppeling ontvangen van bevolkingsonderzoek en wordt opgeroepen voor een intake n.a.v.]"
...,...,...,...,...,...
183,182,20,182_graag hoor_hoor graag_hoor_graag,"[graag hoor, hoor graag, hoor, graag, graag terug, terug, , , , ]","[Ik hoor graag van jullie!, Ik hoor graag van jullie!, Ik hoor graag van jullie!]"
184,183,20,183_koorts koorts_koorts_koorts klein_griepje,"[koorts koorts, koorts, koorts klein, griepje, koorts gelukkig, gelukkig koorts, klein, eetlust, gelukkig, prima]","[Ik heb geen koorts., Ik heb geen koorts., Ik heb geen koorts.]"
185,184,20,184_dankjewel_sorry sorry_ga dank_dankjewel dank,"[dankjewel, sorry sorry, ga dank, dankjewel dank, sorry, buisje ingeleverd, dank krijg, ga zalf, zet spuiten, spuiten ga]","[Dan doe ik dat :) dankjewel., Dankjewel dat zal ik doen ., Dankjewel zal ik doen.]"
186,185,20,185_polibezoek_terugkomend_polibezoek besloten_scopie gaan,"[polibezoek, terugkomend, polibezoek besloten, scopie gaan, scopie vragen, loop moment, doktersverklaring gekregen, afspraak verteld, kom komende, thuis vandaag]","[Ongeveer twee jaar geleden heb ik een doktersverklaring gekregen ivm speciaal toegang in Disneyland Parijs., Vorige week bij het polibezoek van hebben we besloten om de Mri scan te vervroegen naar begin januari., Ontlasting heb ik vandaag ingeleverd, omdat dit 2 weken voor een poli-afspraak moet (mij werd verteld dit eind februari te doen).]"


In [215]:
# Reduce outliers
new_topics = topic_model.reduce_outliers(sentences, topics)

100%|██████████| 20/20 [00:04<00:00,  4.98it/s]


In [216]:
topic_model.update_topics(sentences, topics=new_topics)

2024-11-17 03:29:38,592 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


KeyboardInterrupt: 

In [ ]:
topic_model.get_topic_info()

In [181]:
# hierarchical clustering based on HDBSCAN results
hierarchical_topics = topic_model.hierarchical_topics(sentences)

100%|██████████| 318/318 [00:01<00:00, 175.15it/s]


In [182]:
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:
topics_to_merge = [[194, 123, 28, 24, 213], # GI symptoms
                   [3, 4]] # health update
topic_model.merge_topics(docs, topics_to_merge)

In [186]:
doc_info = topic_model.get_document_info(sentences)
doc_info[doc_info["Topic"] == 70]

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
278,"Ik heb zojuist een dieptepunt gehad, waarbij ik op mijn werk te laat was voor de WC en nu met schone kleren weer aan het werk ben.",70,70_stress_afstuderen_bang_nodige,"[stress, afstuderen, bang, nodige, brengt, invloed, echt, werk, vermoeidheid, werkt]","[Doordat ik aan her afstuderen ben heb ik veel stress door het afstuderen en dit werkt enorm nadelig op mijn darmen., Ik zit er eigenlijk een beetje doorheen met wat ik precies wil doen en heb hier ook van slapeloze nachten van en het brengt de nodige stress met zich mee., Dit brengt de nodige stress met zich mee voor mij, vooral op financieel vlak.]",stress - afstuderen - bang - nodige - brengt - invloed - echt - werk - vermoeidheid - werkt,0.968693,False
606,Ik wil nog graag vijf kilo afvallen en dan zit ik op mijn ideale gewicht.,70,70_stress_afstuderen_bang_nodige,"[stress, afstuderen, bang, nodige, brengt, invloed, echt, werk, vermoeidheid, werkt]","[Doordat ik aan her afstuderen ben heb ik veel stress door het afstuderen en dit werkt enorm nadelig op mijn darmen., Ik zit er eigenlijk een beetje doorheen met wat ik precies wil doen en heb hier ook van slapeloze nachten van en het brengt de nodige stress met zich mee., Dit brengt de nodige stress met zich mee voor mij, vooral op financieel vlak.]",stress - afstuderen - bang - nodige - brengt - invloed - echt - werk - vermoeidheid - werkt,0.999485,False
1720,Hoe nu verder omdat ik super veel last heb?,70,70_stress_afstuderen_bang_nodige,"[stress, afstuderen, bang, nodige, brengt, invloed, echt, werk, vermoeidheid, werkt]","[Doordat ik aan her afstuderen ben heb ik veel stress door het afstuderen en dit werkt enorm nadelig op mijn darmen., Ik zit er eigenlijk een beetje doorheen met wat ik precies wil doen en heb hier ook van slapeloze nachten van en het brengt de nodige stress met zich mee., Dit brengt de nodige stress met zich mee voor mij, vooral op financieel vlak.]",stress - afstuderen - bang - nodige - brengt - invloed - echt - werk - vermoeidheid - werkt,1.000000,False
3273,Maar het infuus heeft niet echt voor verbetering gezorgd ben ik bang.,70,70_stress_afstuderen_bang_nodige,"[stress, afstuderen, bang, nodige, brengt, invloed, echt, werk, vermoeidheid, werkt]","[Doordat ik aan her afstuderen ben heb ik veel stress door het afstuderen en dit werkt enorm nadelig op mijn darmen., Ik zit er eigenlijk een beetje doorheen met wat ik precies wil doen en heb hier ook van slapeloze nachten van en het brengt de nodige stress met zich mee., Dit brengt de nodige stress met zich mee voor mij, vooral op financieel vlak.]",stress - afstuderen - bang - nodige - brengt - invloed - echt - werk - vermoeidheid - werkt,0.934751,False
3455,dat mijn darm te traag werkt op dit moment?,70,70_stress_afstuderen_bang_nodige,"[stress, afstuderen, bang, nodige, brengt, invloed, echt, werk, vermoeidheid, werkt]","[Doordat ik aan her afstuderen ben heb ik veel stress door het afstuderen en dit werkt enorm nadelig op mijn darmen., Ik zit er eigenlijk een beetje doorheen met wat ik precies wil doen en heb hier ook van slapeloze nachten van en het brengt de nodige stress met zich mee., Dit brengt de nodige stress met zich mee voor mij, vooral op financieel vlak.]",stress - afstuderen - bang - nodige - brengt - invloed - echt - werk - vermoeidheid - werkt,0.919932,False
...,...,...,...,...,...,...,...,...
40198,Het begon net goed te gaan met de Crohn maar straks begint het hele verhaal weer van voren af aan omdat ik het infuus niet op tijd heb gehad.,70,70_stress_afstuderen_bang_nodige,"[stress, afstuderen, bang, nodige, brengt, invloed, echt, werk, vermoeidheid, werkt]","[Doordat ik aan her afstuderen ben heb ik veel stress door het afstuderen en dit werkt enorm nadelig op mijn darmen., Ik zit er eigenlijk een beetje doorheen met wat ik precies wil doen en heb hier ook van slapeloze nachten van en het brengt de nodige stress met zich mee., D